# R001: Efficient Hybrid Candidate Generator V1

## 00 — Experiment contract
This notebook executes the R001 candidate generation pipeline. It uses cuML TF-IDF for text fields and exact/rare/numeric matching. DO NOT RUN 50K until 1K smoke passes.

## 01 — GPU environment

In [ ]:
import sys
import os
import subprocess

print(f"Python Version: {sys.version}")
print(f"CWD: {os.getcwd()}")
!nvidia-smi

try:
    import cuml
    import cupy as cp
    print("cuML/CuPy is available. GPU_BACKEND_ACTIVE = TRUE")
except ImportError:
    print("WARNING: cuML/CuPy not found.")


## 02 — GitHub clone/reuse

In [ ]:
import subprocess
REPO_URL = "https://github.com/yugtheguy/amazon_ml.git"
REPO_DIR = "/kaggle/working/amazon_ml"
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists, pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)


## 03 — Commit/version

In [ ]:
!git status --short
!git rev-parse HEAD


## 04 — Dataset discovery

In [ ]:
KAGGLE_DATA_ROOT = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "train_source1.tsv" in files and "train" in root:
        KAGGLE_DATA_ROOT = os.path.dirname(root)
        break

if not KAGGLE_DATA_ROOT:
    KAGGLE_DATA_ROOT = os.path.abspath("data/raw")
print(f"RAW_DATA_ROOT = {KAGGLE_DATA_ROOT}")


## 05 — Processed data prepare/reuse

In [ ]:
if os.path.exists("data/processed/v001/train_source1.parquet"):
    print("Processed data found. Reusing.")
else:
    print("Processed data not found. Checking Kaggle attached datasets...")
    PROCESSED_DATA_ROOT = None
    # Explicit check for the user's known path first
    known_path = "/kaggle/input/datasets/yugdeshmukh/amazon-ml-processed-v001"
    if os.path.exists(os.path.join(known_path, "train_source1.parquet")):
        PROCESSED_DATA_ROOT = known_path
    else:
        for root, dirs, files in os.walk("/kaggle/input"):
            if "train_source1.parquet" in files:
                PROCESSED_DATA_ROOT = root
                break
    
    if PROCESSED_DATA_ROOT:
        print(f"Found attached processed dataset at {PROCESSED_DATA_ROOT}")
        os.makedirs("data/processed/v001", exist_ok=True)
        for f in os.listdir(PROCESSED_DATA_ROOT):
            if f.endswith('.parquet') or f.endswith('.json'):
                os.symlink(os.path.join(PROCESSED_DATA_ROOT, f), os.path.join("data/processed/v001", f))
    else:
        print("No processed dataset attached. Symlinking raw data and building...")
        os.makedirs("data/raw", exist_ok=True)
        if not os.path.exists("data/raw/train") and os.path.exists(os.path.join(KAGGLE_DATA_ROOT, "train")):
            os.symlink(os.path.join(KAGGLE_DATA_ROOT, "train"), "data/raw/train")
        if not os.path.exists("data/raw/test") and os.path.exists(os.path.join(KAGGLE_DATA_ROOT, "test")):
            os.symlink(os.path.join(KAGGLE_DATA_ROOT, "test"), "data/raw/test")
        !PYTHONPATH=. python scripts/build_processed_data.py
        !PYTHONPATH=. python scripts/build_folds.py


## 06 — Tests

In [ ]:
!pip install -r requirements.txt -q
!PYTHONPATH=. python -m pytest tests/ -q


## 07 — Config

In [ ]:
import yaml
import json
with open("configs/candidate_pool_v1.yaml", "r") as f:
    config = yaml.safe_load(f)
print(json.dumps(config, indent=2))


## 08 — 1K smoke

In [ ]:
ARTIFACT_ROOT = "/kaggle/working/artifacts/candidate_pool/R001"
!PYTHONPATH=. python -u scripts/run_r001_candidate_pool.py --data-dir data --processed-dir /kaggle/input/datasets/yugdeshmukh/amazon-ml-processed-v001 --smoke-size 1000 --probe-size 0 --out-dir /kaggle/working/artifacts/candidate_pool/R001/smoke_1k


## 09 — Smoke metrics

In [ ]:
if os.path.exists("/kaggle/working/artifacts/candidate_pool/R001/smoke_1k/retrieval_metrics.json"):
    with open("/kaggle/working/artifacts/candidate_pool/R001/smoke_1k/retrieval_metrics.json") as f:
        print(json.dumps(json.load(f), indent=2))


## 10 — 50K run
**RUN ONLY AFTER 1K SMOKE PASSES**

In [ ]:
!PYTHONPATH=. python -u scripts/run_r001_candidate_pool.py --data-dir data --processed-dir /kaggle/input/datasets/yugdeshmukh/amazon-ml-processed-v001 --smoke-size 0 --probe-size 50000 --out-dir /kaggle/working/artifacts/candidate_pool/R001/probe_50k


## 11 — Candidate frontier

In [ ]:
import pandas as pd
f = "/kaggle/working/artifacts/candidate_pool/R001/probe_50k/candidate_count_vs_recall.csv"
if os.path.exists(f):
    display(pd.read_csv(f))


## 12 — Channel contributions

In [ ]:
f2 = "/kaggle/working/artifacts/candidate_pool/R001/probe_50k/channel_contribution.csv"
if os.path.exists(f2):
    display(pd.read_csv(f2))


## 13 — Miss analysis

In [ ]:
f3 = "/kaggle/working/artifacts/candidate_pool/R001/probe_50k/missed_gt_pairs.parquet"
if os.path.exists(f3):
    missed = pd.read_parquet(f3)
    print(f"Total Missed GT pairs: {len(missed)}")
    display(missed.head())


## 14 — Artifact packaging

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/R001_smoke_artifacts", 'zip', "/kaggle/working/artifacts/candidate_pool/R001/smoke_1k")
print("Artifacts packaged to /kaggle/working/R001_smoke_artifacts.zip")
